In [2]:
!git clone https://github.com/MLSA-SRM/recruit-task-rag-docs.git

Cloning into 'recruit-task-rag-docs'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 0), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), done.


In [3]:
from pathlib import Path

docs_path = Path("recruit-task-rag-docs")

documents = []

for filename in [
    "01-getting-started.md",
    "02-pricing-and-plans.md",
    "03-troubleshooting.md"
]:
    path = docs_path / filename

    documents.append({
        "source": filename,
        "text": path.read_text()
    })

print(f"Loaded {len(documents)} documents")

for doc in documents:
    print(doc["source"], "→", len(doc["text"]), "characters")

Loaded 3 documents
01-getting-started.md → 1571 characters
02-pricing-and-plans.md → 1380 characters
03-troubleshooting.md → 1798 characters


In [ ]:
import re

def chunk_document(text, source):
    sections = re.split(r"\n(?=## )", text)

    chunks = []

    for i, section in enumerate(sections):
        section = section.strip()

        if section:
            chunks.append({
                "text": section,
                "source": source,
                "chunk_id": i
            })

    return chunks

In [5]:
all_chunks = []

for doc in documents:
    chunks = chunk_document(doc["text"], doc["source"])
    all_chunks.extend(chunks)


In [6]:
!pip install -q sentence-transformers

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
for chunk in all_chunks:
    chunk["embedding"] = model.encode(chunk["text"])


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
def retrieve(query, top_k=3, threshold=0.30):
    query_embedding = model.encode(query)

    results = []

    for chunk in all_chunks:
        score = cosine_similarity(
            [query_embedding],
            [chunk["embedding"]]
        )[0][0]

        results.append({
            "text": chunk["text"],
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "score": score
        })

    results.sort(key=lambda x: x["score"], reverse=True)

    # Check whether the best result is relevant enough
    if results[0]["score"] < threshold:
        return []

    return results[:top_k]

In [11]:
!pip install -q transformers sentencepiece

In [12]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.0 MB/s eta 0:00:00


In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("Qwen loaded!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded!


In [14]:
def answer_question(question):
    # 1. Retrieve relevant passages
    results = retrieve(question, top_k=1)

    # 2. If nothing relevant was found
    if not results:
        return {
            "answer": "I don't know based on the provided documents.",
            "sources": []
        }

    # 3. Combine retrieved passages
    context = "\n\n".join(
        result["text"] for result in results
    )

    # 4. Create a grounded prompt
    prompt = f"""
Answer the question using ONLY the information in the passages below.

If the passages do not contain the answer, say:
"I don't know based on the provided documents."

Passages:
{context}

Question:
{question}

Answer:
"""

    # 5. Send prompt to Qwen
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_prompt,
        return_tensors="pt"
    ).to(llm.device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    # 6. Return answer + source information
    sources = [
        {
            "source": result["source"],
            "chunk_id": result["chunk_id"],
            "score": result["score"]
        }
        for result in results
    ]

    return {
        "answer": answer,
        "sources": sources
    }

In [15]:
def display_answer(result):
    print("🤖 ANSWER")
    print(result["answer"])

    if result["sources"]:
        print("\n📚 SOURCES")

        for source in result["sources"]:
            print("\n" + "=" * 60)
            print(f"Document: {source['source']}")
            print(f"Chunk: {source['chunk_id']}")
            print(f"Similarity: {source['score']:.3f}")

            # Find the original chunk
            for chunk in all_chunks:
                if (
                    chunk["source"] == source["source"]
                    and chunk["chunk_id"] == source["chunk_id"]
                ):
                    print("\nPassage:")
                    print(chunk["text"])
                    break
    else:
        print("\n📚 SOURCES")
        print("No relevant passages found.")

In [16]:
# testing the previous code to check the output:
result = answer_question(
    "How often does NimbusNote sync?"
)

display_answer(result)

🤖 ANSWER
NimbusNote syncs every 15 seconds while the app is in the foreground, and every 5 minutes in the background.

📚 SOURCES

Document: 01-getting-started.md
Chunk: 3
Similarity: 0.721

Passage:
## Sync behavior

NimbusNote syncs every 15 seconds while the app is in the foreground, and every 5 minutes in the background. If two devices edit the same note within that sync window, NimbusNote keeps both versions as separate note revisions rather than silently merging them — the user is asked to pick which version to keep the next time they open the note.


In [18]:
while True:
    question = input("\nAsk a question (or type 'exit'): ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    result = answer_question(question)
    display_answer(result)


Ask a question (or type 'exit'): "How much does the Pro plan cost?"
🤖 ANSWER
The Pro plan costs $120 per month.

📚 SOURCES

Document: 02-pricing-and-plans.md
Chunk: 6
Similarity: 0.513

Passage:
## Student discount

Workspaces verified as belonging to a student (via a `.edu` email or equivalent) get 50% off the Pro plan. There is currently no student discount on the Team plan.

Ask a question (or type 'exit'): What is the capital of France?
🤖 ANSWER
I don't know based on the provided documents.

📚 SOURCES
No relevant passages found.

Ask a question (or type 'exit'): exit
Goodbye!
